The goal of this notebook is to investigate and quantify the change in each Copernicus plankton group over the lifetimes of cyclones traveling South and anticyclones traveling North respectively. The groups are six phytoplankton functional types (diatoms, dinophytes, green algae, haptophytes, prochlorophytes, prokaryotes) and three size classes (micro, nano, pico). Each group field is the chlorophyll-a concentration attributed to that group, in mg/m³.

Target eddies and eddy requirements are the same as in `chl_over_lifetime.ipynb`:
- Cyclones formed north of the Gulf Stream axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south. Anticyclones use the reversed rule.
- 50% CHL coverage from CMEMS and at least 10 valid pixels per eddy-composite.
- A group value keeps the same rule on its own pixels, because the group mask is smaller than the CHL mask.

Two views:
- Over lifetime: the interior mean of each group per age bin, one value per eddy, like the third figure of `chl_over_lifetime.ipynb`.
- Age and radius: the mean of each group in each age bin and each 0.2 R ring from the eddy center out to 2 radii. The rings are circles around the track center nearest the composite midpoint, R is the PET speed radius, and the pixels come from the same 8-day composites. A ring needs 3 valid pixels and 50% coverage, and a cell needs 3 eddies.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']
groups = ['DIATO', 'DINO', 'GREEN', 'HAPTO', 'PROCHLO', 'PROKAR', 'MICRO', 'NANO', 'PICO']
group_labels = {
    'DIATO': 'Diatoms', 'DINO': 'Dinophytes', 'GREEN': 'Green algae', 'HAPTO': 'Haptophytes',
    'PROCHLO': 'Prochlorophytes', 'PROKAR': 'Prokaryotes',
    'MICRO': 'Microphytoplankton', 'NANO': 'Nanophytoplankton', 'PICO': 'Picophytoplankton',
}
panel_letters = 'abcdefghi'

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
plankton = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
plankton = plankton.merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']],
    on=identity_columns, how='left',
)
target_plankton = cast(pd.DataFrame, plankton.loc[plankton['is_target']]).copy()
if EXCLUDE_RECORD_EDGE_TRACKS:
    target_plankton = cast(pd.DataFrame, target_plankton.loc[~target_plankton['at_record_edge']]).copy()
plankton_settings = cfg['collocate_plankton']
for group in groups:
    covered = (
        target_plankton[f'{group}_n_pixels'].ge(plankton_settings['min_pixels'])
        & target_plankton[f'{group}_n_pixels'].div(target_plankton['n_pixels']).ge(plankton_settings['min_coverage'])
    )
    target_plankton.loc[~covered, group] = np.nan

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

print(f'Target eddies and composites in the Copernicus plankton table: the distinct eddies of each polarity, the eddy-composites that pass the CHL rule of {plankton_settings["min_coverage"]:.0%} coverage of the speed-contour interior and {plankton_settings["min_pixels"]} valid pixels, and the eddy-composites that keep each group after the same rule on the pixels of that group, which are fewer because the group mask is smaller than the CHL mask. {len(target_plankton.drop_duplicates(identity_columns))} of the {int(eddy_tracks["is_target"].sum())} target eddies have at least one composite.')
display(cast(pd.DataFrame, target_plankton.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    **{group: (group, 'count') for group in groups},
)))

In [ ]:
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
analysis = target_plankton.sort_values(identity_columns + ['date']).melt(
    id_vars=identity_columns + ['date', 'age_frac'], value_vars=groups,
    var_name='group', value_name='concentration',
)
analysis['age_bin'] = np.minimum(
    (analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['group'])['concentration'].transform('first')
eddy_bins = cast(pd.DataFrame, analysis.groupby(identity_columns + ['group', 'age_bin']).agg(
    concentration=('concentration', 'mean'), change=('change', 'mean'),
)).reset_index()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for polarity in polarity_names:
    polarity_bins = eddy_bins.loc[eddy_bins['polarity'].eq(polarity)]
    eddy_ids = sorted(polarity_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for group in groups:
        group_bins = polarity_bins.loc[polarity_bins['group'].eq(group)]
        for metric in ('concentration', 'change'):
            matrix = group_bins.pivot(index='track_id', columns='age_bin', values=metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
            counts = np.isfinite(matrix).sum(axis=0)
            means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
            sampled = matrix[draws]
            sampled_counts = np.isfinite(sampled).sum(axis=1)
            sampled_means = np.divide(
                np.nansum(sampled, axis=1), sampled_counts,
                out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0,
            )
            for age_bin in range(N_AGE_BINS):
                bootstrap_values = sampled_means[:, age_bin]
                bootstrap_values = bootstrap_values[np.isfinite(bootstrap_values)]
                low, high = (np.quantile(bootstrap_values, [0.025, 0.975]) if counts[age_bin] >= 3 else (np.nan, np.nan))
                summary_rows.append({
                    'polarity': polarity, 'group': group, 'metric': metric, 'age_bin': age_bin,
                    'age_midpoint': bin_centers[age_bin], 'mean': means[age_bin],
                    'ci_low': low, 'ci_high': high, 'n_eddies': int(counts[age_bin]),
                })
lifetime_summary = pd.DataFrame(summary_rows)
n_eddies = analysis.groupby('polarity')['track_id'].nunique()
bin_counts = lifetime_summary.loc[lifetime_summary['metric'].eq('concentration')].groupby(['polarity', 'age_bin'])['n_eddies'].min().unstack('age_bin')

for metric, ylabel, opening in (
    ('concentration', 'Group chlorophyll-a (mg m$^{-3}$)', f'Interior Copernicus concentration of each plankton group in the target eddies against the fraction of the observed track, the time from the first to the last detection in {N_AGE_BINS} equal bins, one panel per group. A point is the mean of the group inside the speed contour.'),
    ('change', 'Change from first observation (mg m$^{-3}$)', f'Change of the interior Copernicus concentration of each plankton group from the first composite of the same target eddy, against the fraction of the observed track in {N_AGE_BINS} equal bins, one panel per group. The gray line marks no change.'),
):
    life_fig, life_axes = cast(tuple[Figure, np.ndarray], plt.subplots(3, 3, figsize=(7.2, 6.6), sharex=True, layout='constrained'))
    for letter, ax, group in zip(panel_letters, life_axes.flat, groups):
        ax = cast(Axes, ax)
        if metric == 'change':
            ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
        for polarity in polarity_names:
            result = lifetime_summary.loc[
                lifetime_summary['polarity'].eq(polarity) & lifetime_summary['group'].eq(group) & lifetime_summary['metric'].eq(metric)
            ].sort_values('age_bin')
            color = polarity_colors[polarity]
            intervals = result.loc[result['ci_low'].notna()]
            ax.errorbar(
                intervals['age_midpoint'], intervals['mean'],
                yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
                fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2,
            )
            ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3.2, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[polarity]})', zorder=3)
        ax.set_title(f'$\\bf{{({letter})}}$ {group_labels[group]}', loc='left')
        ax.xaxis.set_ticks(np.linspace(0, 1, 6))
        ax.set_xlim(0, 1)
        ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
        ax.yaxis.set_major_locator(locator)
        ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
        ax.set_ylim(ticks[0], ticks[-1])
    for ax in life_axes[-1]:
        cast(Axes, ax).set_xlabel('Fraction of observed track')
    life_fig.supylabel(ylabel, fontsize=8)
    life_fig.legend(*cast(Axes, life_axes.flat[0]).get_legend_handles_labels(), loc='outside lower center', ncol=2)
    plt.show()
    print(f'{opening} Each group value is the chlorophyll-a the Copernicus product attributes to that group, and nanophytoplankton equals haptophytes in this product, so panels (d) and (h) repeat. Within a bin the composites of one eddy are averaged first, so each eddy counts once per bin, and a bin an eddy never reaches stays empty. Points are the mean over eddies and error bars the 95% interval from {N_BOOTSTRAP} resamplings of the eddies, drawn where at least 3 eddies contribute. Target cyclones: {n_eddies["cyclone"]} eddies, at least {bin_counts.loc["cyclone"].min()} in every bin of every group. Target anticyclones: {n_eddies["anticyclone"]} eddies, at least {bin_counts.loc["anticyclone"].min()} in every bin of every group. A group value enters only where the group covers {plankton_settings["min_coverage"]:.0%} of the interior pixels with at least {plankton_settings["min_pixels"]} of them.')
print('Mean group concentration per bin, the values of the first figure, in mg/m³.')
display(lifetime_summary.loc[lifetime_summary['metric'].eq('concentration')].pivot(index='group', columns=['polarity', 'age_bin'], values='mean').reindex(groups).round(4))
print('Eddies per bin, the minimum over the nine groups.')
display(bin_counts)

In [ ]:
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
observations = []
for polarity in polarity_names:
    tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
    observations.append(pd.DataFrame({
        'polarity': polarity, 'track_id': tracked.track.astype(int),
        'day': pd.to_datetime(tracked.time.astype(int), unit='D', origin=pd.Timestamp('1950-01-01')),
        'center_lon': (tracked.longitude + 180) % 360 - 180, 'center_lat': tracked.latitude,
        'radius_km': tracked.radius_s / 1000, 'virtual': tracked.virtual.astype(bool),
    }))
observations = pd.concat(observations, ignore_index=True)
observations = observations.loc[~observations['virtual']].merge(target_plankton[identity_columns].drop_duplicates(), on=identity_columns)
windows = []
for year in range(physical_start.year, physical_end.year + 1):
    start = pd.Timestamp(year, 1, 1)
    while start.year == year:
        end = min(start + pd.Timedelta(days=7), pd.Timestamp(year, 12, 31))
        if end >= physical_start and start <= physical_end:
            windows.append((start, end))
        start = end + pd.Timedelta(days=1)
fields = xr.open_mfdataset(sorted((DATA_DIR / 'bronze/plankton').glob('plankton_*.nc')), combine='by_coords')[groups]
lon = fields['longitude'].to_numpy()
lat = fields['latitude'].to_numpy()
rings = []
for start, end in windows:
    composites = target_plankton.loc[target_plankton['date'].between(start, end), identity_columns + ['date', 'age_frac']]
    if composites.empty:
        continue
    candidates = observations.loc[observations['day'].between(start, end)].merge(composites, on=identity_columns)
    candidates['offset'] = (candidates['day'] - candidates['date']).abs()
    composite = fields.sel(time=slice(start, end)).mean('time').load()
    for eddy in candidates.sort_values(['offset', 'day']).drop_duplicates(identity_columns).itertuples():
        half_width = MAX_RADIUS * eddy.radius_km / 111.32
        lon_index = np.flatnonzero(np.abs(lon - eddy.center_lon) <= half_width / np.cos(np.radians(eddy.center_lat)))
        lat_index = np.flatnonzero(np.abs(lat - eddy.center_lat) <= half_width)
        lon_grid, lat_grid = np.meshgrid(lon[lon_index], lat[lat_index])
        half_chord = (
            np.sin(np.radians(lat_grid - eddy.center_lat) / 2) ** 2
            + np.cos(np.radians(lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(lon_grid - eddy.center_lon) / 2) ** 2
        )
        distance_km = 2 * 6371 * np.arcsin(np.sqrt(half_chord))
        radial_bin = np.digitize(distance_km.ravel() / eddy.radius_km, radial_edges) - 1
        inside = radial_bin < N_RADIAL_BINS
        n_pixels = np.bincount(radial_bin[inside], minlength=N_RADIAL_BINS)
        ring = {
            'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': eddy.date, 'age_frac': eddy.age_frac,
            'radial_bin': np.arange(N_RADIAL_BINS), 'n_pixels': n_pixels,
        }
        box = composite.isel(longitude=lon_index, latitude=lat_index)
        for group in groups:
            values = box[group].to_numpy().ravel()
            valid = inside & np.isfinite(values)
            n_valid = np.bincount(radial_bin[valid], minlength=N_RADIAL_BINS)
            covered = (n_valid >= 3) & (n_valid >= plankton_settings['min_coverage'] * n_pixels)
            ring[group] = np.where(covered, np.bincount(radial_bin[valid], weights=values[valid], minlength=N_RADIAL_BINS) / np.maximum(n_valid, 1), np.nan)
            ring[f'{group}_n_pixels'] = n_valid
        rings.append(pd.DataFrame(ring))
rings = pd.concat(rings, ignore_index=True)
radial_analysis = rings.melt(
    id_vars=identity_columns + ['date', 'age_frac', 'radial_bin'], value_vars=groups,
    var_name='group', value_name='concentration',
)
radial_analysis['age_bin'] = np.minimum(
    (radial_analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
eddy_cells = radial_analysis.groupby(identity_columns + ['group', 'age_bin', 'radial_bin'])['concentration'].mean().reset_index()
cells = cast(pd.DataFrame, eddy_cells.groupby(['polarity', 'group', 'age_bin', 'radial_bin']).agg(
    concentration=('concentration', 'mean'), n_eddies=('concentration', 'count'),
)).reset_index()
cells.loc[cells['n_eddies'].lt(3), 'concentration'] = np.nan
cell_counts = cells.groupby(['radial_bin', 'polarity', 'age_bin'])['n_eddies'].min().unstack(['polarity', 'age_bin'])

cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('#e6e6e6')
radial_fig = plt.figure(figsize=(7.2, 5.8))
outer = radial_fig.add_gridspec(3, 3, left=0.075, right=0.94, bottom=0.075, top=0.935, wspace=0.6, hspace=0.6)
for index, (letter, group) in enumerate(zip(panel_letters, groups)):
    group_cells = cells.loc[cells['group'].eq(group)]
    vmin, vmax = group_cells['concentration'].min(), group_cells['concentration'].max()
    axes = cast(np.ndarray, outer[index].subgridspec(1, 2, wspace=0.12).subplots(sharey=True))
    for ax, polarity in zip(axes, polarity_names):
        ax = cast(Axes, ax)
        grid = group_cells.loc[group_cells['polarity'].eq(polarity)].pivot(index='radial_bin', columns='age_bin', values='concentration').reindex(index=range(N_RADIAL_BINS), columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=cmap, vmin=vmin, vmax=vmax, edgecolors='white', linewidth=0.3)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        ax.set_aspect('equal')
        ax.set_title(f'{polarity.capitalize()}s', fontsize=7, pad=3)
        ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(bin_edges, minor=True)
        ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
        ax.yaxis.set_ticks(radial_edges, minor=True)
        ax.tick_params(labelsize=6.5, length=2)
        ax.tick_params(which='minor', length=1.2)
    colorbar = radial_fig.colorbar(ScalarMappable(norm=Normalize(vmin, vmax), cmap=cmap), cax=cast(Axes, axes[1]).inset_axes((1.12, 0, 0.1, 1)))
    colorbar.ax.tick_params(labelsize=6.5, length=2)
    colorbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
    colorbar.outline.set_linewidth(0.5)
    cast(Axes, axes[0]).text(0, 1.14, f'$\\bf{{({letter})}}$ {group_labels[group]}', transform=axes[0].transAxes, fontsize=8.5, va='bottom', ha='left')
radial_fig.supxlabel('Fraction of observed track', fontsize=8)
radial_fig.supylabel('Distance from eddy center (speed radii)', fontsize=8)
plt.show()
print(f'Copernicus concentration of each plankton group by distance from the eddy center and fraction of the observed track, one panel pair per group, with the target cyclones on the left and the target anticyclones on the right of each pair on one color scale in mg/m³. Each column is an age bin of {bin_edges[1]:.1f} of the track and each row a ring {radial_edges[1]:.1f} speed radii wide around the track center nearest the composite midpoint, out to {MAX_RADIUS} radii. The dashed line is the speed contour, at one radius. A ring value is the mean of the composite pixels of the group in the ring, taken where the group covers {plankton_settings["min_coverage"]:.0%} of the ring with at least 3 pixels, averaged first within each eddy and age bin and then over eddies, so each eddy counts once per cell. A cell needs 3 eddies and is gray otherwise. {len(rings.drop_duplicates(identity_columns + ["date"]))} composites of {len(rings.drop_duplicates(identity_columns))} eddies enter, and every cell holds at least {cell_counts["cyclone"].min().min()} cyclones and {cell_counts["anticyclone"].min().min()} anticyclones. Nanophytoplankton equals haptophytes in this product, so panels (d) and (h) repeat.')
print('Eddies per cell, the minimum over the nine groups, by ring (rows) and age bin (columns).')
display(cell_counts)